In [5]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
words = open('names.txt', 'r').read().splitlines()
words[:2]

['emma', 'olivia']

In [7]:
len(words)

32033

In [8]:
#Build the vocabulary of characters and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [9]:
# Building The dataset

block_size = 3 # context length : how many characters we take to predict the next one

X, Y = [], []
for w in words[:1]:
    print(w)
    
    # context = [0]
    # print(context)
    
    context = [0] * block_size
    # print(context)

    # x = w + '.'
    
    # print(x)
    
    for ch in w + '.':
        ix = stoi[ch] # 5
        X.append(context)
        # print(X)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '------>', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)
        

emma
... ------> e
..e ------> m
.em ------> m
emm ------> a
mma ------> .


In [10]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1]])

In [11]:
Y

tensor([ 5, 13, 13,  1,  0])

In [12]:
seed = torch.Generator().manual_seed(123456)

In [13]:
# Embedding Table / Glorified Lookup table
C = torch.randn((27,2), generator=seed)
C

tensor([[-1.1218, -1.8445],
        [ 1.0258,  2.1921],
        [-1.4322,  1.5293],
        [ 1.2118,  0.0423],
        [ 0.3807,  0.0538],
        [ 0.5047, -0.4987],
        [-0.4891,  0.0578],
        [ 0.9436,  0.9775],
        [ 0.7843, -0.1034],
        [ 0.1187, -0.2737],
        [ 1.1537,  1.3910],
        [ 1.0732, -1.2210],
        [ 0.9103,  0.8257],
        [ 0.1247,  0.8079],
        [ 0.1415, -0.5531],
        [ 1.0828, -1.7865],
        [-0.7372, -0.7429],
        [-0.4891,  2.4613],
        [-0.7179, -0.2446],
        [-0.8818,  1.6565],
        [-0.5419, -2.4956],
        [ 0.3731,  1.2806],
        [-0.1539,  0.2954],
        [-1.5935, -0.0451],
        [-0.2014, -0.7450],
        [-0.2250,  0.0894],
        [-1.0029, -1.1601]])

In [14]:
C[5]

tensor([ 0.5047, -0.4987])

In [15]:
emb = C[X]
emb.shape

torch.Size([5, 3, 2])

In [16]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [26]:
emb

tensor([[[-1.1218, -1.8445],
         [-1.1218, -1.8445],
         [-1.1218, -1.8445]],

        [[-1.1218, -1.8445],
         [-1.1218, -1.8445],
         [ 0.5047, -0.4987]],

        [[-1.1218, -1.8445],
         [ 0.5047, -0.4987],
         [ 0.1247,  0.8079]],

        [[ 0.5047, -0.4987],
         [ 0.1247,  0.8079],
         [ 0.1247,  0.8079]],

        [[ 0.1247,  0.8079],
         [ 0.1247,  0.8079],
         [ 1.0258,  2.1921]]])

In [34]:
torch.cat((emb[:,0,:],emb[:,1,:],emb[:,2,:]),dim=1)

tensor([[-1.1218, -1.8445, -1.1218, -1.8445, -1.1218, -1.8445],
        [-1.1218, -1.8445, -1.1218, -1.8445,  0.5047, -0.4987],
        [-1.1218, -1.8445,  0.5047, -0.4987,  0.1247,  0.8079],
        [ 0.5047, -0.4987,  0.1247,  0.8079,  0.1247,  0.8079],
        [ 0.1247,  0.8079,  0.1247,  0.8079,  1.0258,  2.1921]])

In [30]:
torch.cat(torch.unbind(emb,1),1)

tensor([[-1.1218, -1.8445, -1.1218, -1.8445, -1.1218, -1.8445],
        [-1.1218, -1.8445, -1.1218, -1.8445,  0.5047, -0.4987],
        [-1.1218, -1.8445,  0.5047, -0.4987,  0.1247,  0.8079],
        [ 0.5047, -0.4987,  0.1247,  0.8079,  0.1247,  0.8079],
        [ 0.1247,  0.8079,  0.1247,  0.8079,  1.0258,  2.1921]])

In [ ]:
a = torch.arange(18)
a

In [ ]:
a.shape

In [ ]:
a.view(3,3,2)

In [ ]:
a.storage()

In [ ]:
emb.storage()

In [ ]:
emb.view(5,6)

In [ ]:
h = emb.view(-1,6) @ W1 + b1

In [ ]:
h

In [ ]:
h = torch.tanh(h)

In [ ]:
h.shape

In [ ]:
W2 = torch.randn((100,27), generator=seed)
b2 = torch.randn(27, generator=seed)

In [ ]:
logits = h @ W2 + b2
logits

In [ ]:
logits.shape

In [ ]:
counts = logits.exp()

In [ ]:
probs = counts / counts.sum(1, keepdims=True)
probs

In [ ]:
probs.shape

In [ ]:
probs[0].sum()

In [ ]:
loss = -probs[torch.arange(5),Y].log().mean()
loss

In [ ]:
# Model

In [ ]:
X.shape, Y.shape

In [ ]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2),generator=g, requires_grad=True)
W1 = torch.randn((6,100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100,27), generator=g, requires_grad=True)
b2 = torch.randn(27,generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]

In [ ]:
sum(p.nelement() for p in parameters)

In [ ]:
learning_rate = -0.1

In [ ]:
for _ in range(10):
    # Forward Pass
    xemb = C[X] # 5,3,2
    # --- Neuron ----
    h = xemb.view(-1,6) @ W1 + b1
    h = torch.tanh(h)
    
    # --- Loss Calc ----
    logits = h @ W2 + b2
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdims=True)
    # loss = -probs[torch.arange(5),Y].log().mean()
    loss = F.cross_entropy(logits,Y)
    print(loss.item())
    
    # Back Prop
    for p in parameters:
        p.grad = None
    loss.backward()
    
    # Update
    for p in parameters:
        p.data += -0.01 * p.grad

In [ ]:
logits.max(1)

# MLP Explained

In [ ]:
# === Building a dataset ===
xs = []
ys = []

block_size = 3
content = [0] * 3

for w in words[:1]: 
    for ch in w + '.':
        ix = stoi[ch]
        xs.append(content)
        ys.append(ix)
        content = content[1:] + [ix]

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [ ]:
xs,ys

In [ ]:
seedz = torch.Generator().manual_seed(2147483647) # For making it deterministic

In [ ]:
# === LAYER BUILDING ===
Cz = torch.randn((27,2),generator=seedz, requires_grad=True) # embedding Matrix

In [ ]:
# ==== Embedding + Foward Pass ===
xs_embedded = Cz[xs]

In [ ]:
# === 1st layer ===
# XS contains a shape of [...,3(context window),2(embedding vectors)]
W1 = torch.randn((6,100), generator=seedz, requires_grad=True) # weights matrix
b1 = torch.randn((100), generator=seedz, requires_grad=True) # bias matrix

In [ ]:
# === Defining Neurons in 1st Layer + Forward Pass ===
h = xs_embedded.view(-1,6) @ W1 + b1
h = torch.tanh(h)

In [ ]:
# === 2nd Layer ===

W2 = torch.randn((100,27), generator=seedz, requires_grad=True)
b2 = torch.randn((27), generator=seedz, requires_grad=True)

In [ ]:
parameters = [Cz, W1, b1, W2, b2]

In [ ]:
# === Defining Neurons in 2nd Layer + Forward Pass ===
logits = h @ W2 + b2

In [ ]:
# === Loss Calculation
loss = F.cross_entropy(logits, ys)
print(loss.item())

In [ ]:
# === Backpropagation ===
for p in parameters:
    p.grad = None
    
loss.backward()

In [ ]:
# === Updating Data
for p in parameters:
    p.data += -0.1 * p.grad

# GPU Bigram MLP

In [ ]:
device = "cuda" if torch.cuda.is_available else "cpu"
device

In [ ]:
def build_dataset(words):
    block_size = 3
    gx = []
    gy = []
    
    for w in words:
        content = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            gx.append(content)
            gy.append(ix)
            content = content[1:] + [ix]

    gx = torch.tensor(gx).to(device)
    gy = torch.tensor(gy).to(device)
    print(gx.shape,gy.shape)
    return gx,gy

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev= build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [ ]:
Xtr.shape, Ytr.shape

In [ ]:
gg = torch.Generator(device=device).manual_seed(2147483647)

In [ ]:
gC = torch.randn((27,17), generator=g).to(device)
gW1 = torch.randn((51,579), generator=g).to(device)
gb1 = torch.randn((579), generator=g).to(device)
gW2 = torch.randn((579,27), generator=g).to(device)
gb2 = torch.randn((27), generator=g).to(device)
parameters = [gC, gW1, gb1, gW2, gb2]

In [ ]:
sum(p.nelement() for p in parameters)

In [ ]:
for p in parameters:
    p.requires_grad = True

In [ ]:
lre = torch.linspace(-3,0,1000)
lrs = 10**lre

In [ ]:
lri = []
lossi = []
stepi = []

In [ ]:
for i in range(100000):

    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (43,))
    
    # Forward Pass
    gx_emb = gC[Xtr[ix]]
    h = torch.tanh(gx_emb.view(-1,51) @ gW1 + gb1)
    logits = h @ gW2 + gb2
    loss = F.cross_entropy(logits,Ytr[ix])
    #print(loss.item())
    
    # Back Propagation
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update
    lr = 0.001 if i < 50000 else 0.0001
    for p in parameters:
        p.data += -lr * p.grad

    #track stats
    # lri.append(lrs[i])
    stepi.append(i)
    lossi.append(loss.log10().item())
#print(loss.item())

In [ ]:
print(loss.item())

In [ ]:
plt.plot(stepi,lossi)

In [ ]:
gx_emb = gC[Xtr]
h = torch.tanh(gx_emb.view(-1,51) @ gW1 + gb1)
logits = h @ gW2 + gb2
loss = F.cross_entropy(logits,Ytr)
print(loss.item())

In [ ]:
gx_emb = gC[Xdev]
h = torch.tanh(gx_emb.view(-1,51) @ gW1 + gb1)
logits = h @ gW2 + gb2
loss = F.cross_entropy(logits,Ydev)
print(loss.item())

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(gC[:,0].to(device='cpu').data,gC[:,1].to(device='cpu').data, s=200)
for i in range(gC.shape[0]):
    plt.text(gC[i,0].to(device='cpu').item(),gC[i,1].to(device='cpu').item(),itos[i],ha="center",va="center",color="white")
plt.grid('minor')

In [ ]:
# training split, dev/validation split, test split.
# 80% , 10%, 10%

# Sampling

In [ ]:
for _ in range(20):

    out = []
    context = [0] * block_size

    while True:
        gemb = gC[torch.tensor([context])]
        h = torch.tanh(gemb.view(1,-1) @ gW1 + gb1)
        logits = h @ gW2 + gb2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1,generator=gg).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix ==0:
            break

    print(''.join(itos[i] for i in out))